In [36]:
# =========================
# 1. Data Load + Time Split + Train-based MinMax Scaling
# =========================
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.preprocessing import MinMaxScaler

from scipy.stats import chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# 데이터 불러오기
df = pd.read_csv(r"../../../data/processed/data_vif.csv")

target = "Risk_Label"
date_col = "Date"

# 날짜 기준 정렬
if date_col in df.columns:
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)

# =========================
# Risk_Label 0/1 변환
# =========================

print("Risk_Label 변환 전 unique:")
print(df[target].unique())

df[target] = df[target].astype(str).str.strip()

df[target] = df[target].replace({
    "Low Risk": 0,
    "High Risk": 1,
    "Low": 0,
    "High": 1,
    "0": 0,
    "1": 1,
    "0.0": 0,
    "1.0": 1
})

print("\nRisk_Label 변환 후 unique:")
print(df[target].unique())

df[target] = df[target].astype(int)

# Date 제거
if date_col in df.columns:
    df = df.drop(columns=[date_col])

# X, y 분리
X = df.drop(columns=[target])
y = df[target]

# 비수치형 컬럼 확인
non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric_cols) > 0:
    raise ValueError(f"비수치형 컬럼이 있습니다. 먼저 인코딩 필요: {non_numeric_cols}")

# 시간순 train / valid / test = 5 : 3 : 2
n_total = len(df)
n_train = int(n_total * 0.5)
n_valid = int(n_total * 0.3)

train_end = n_train
valid_end = n_train + n_valid

X_train = X.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()

X_valid = X.iloc[train_end:valid_end].copy()
y_valid = y.iloc[train_end:valid_end].copy()

X_test = X.iloc[valid_end:].copy()
y_test = y.iloc[valid_end:].copy()

print("Split size")
print("train:", X_train.shape, y_train.shape)
print("valid:", X_valid.shape, y_valid.shape)
print("test :", X_test.shape, y_test.shape)

print("\nClass ratio")
print("train:", y_train.value_counts(normalize=True).to_dict())
print("valid:", y_valid.value_counts(normalize=True).to_dict())
print("test :", y_test.value_counts(normalize=True).to_dict())

# =========================
# Train 기준 Min-Max Scaling
# =========================

scaler = MinMaxScaler()

scaler.fit(X_train)

X_train_scaled = pd.DataFrame(
    scaler.transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_valid_scaled = pd.DataFrame(
    scaler.transform(X_valid),
    columns=X_valid.columns,
    index=X_valid.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("\nScaling check")
print("train min/max:", X_train_scaled.min().min(), X_train_scaled.max().max())
print("valid min/max:", X_valid_scaled.min().min(), X_valid_scaled.max().max())
print("test  min/max:", X_test_scaled.min().min(), X_test_scaled.max().max())

Risk_Label 변환 전 unique:
<StringArray>
['Low Risk', 'High Risk']
Length: 2, dtype: str

Risk_Label 변환 후 unique:
[0 1]
Split size
train: (2054, 28) (2054,)
valid: (1232, 28) (1232,)
test : (822, 28) (822,)

Class ratio
train: {0: 0.9050632911392406, 1: 0.0949367088607595}
valid: {0: 0.8685064935064936, 1: 0.1314935064935065}
test : {0: 0.8880778588807786, 1: 0.11192214111922141}

Scaling check
train min/max: 0.0 1.0000000000000002
valid min/max: -0.6758599166295274 2.0623447590660704
test  min/max: -0.24909554059542227 1.698708395429707


In [37]:
# =========================
# Logistic inference-based Forward / Backward / Stepwise Selection
# =========================
signal_prefixes = ["Signal1", "Signal2", "Signal3", "Signal4"]


def feature_to_unit(feature):
    """
    Signal4_Buy, Signal4_Sell, Signal4_Stay 같은 변수는
    하나의 선택 단위 Signal4로 묶는다.
    """
    for sig in signal_prefixes:
        if feature.startswith(sig + "_"):
            return sig
    return feature


def make_selection_units(X_columns):
    """
    일반 변수는 그대로 하나의 unit.
    Signal 변수는 Signal1~Signal4 그룹 단위로 묶음.
    """
    units = []

    for col in X_columns:
        unit = feature_to_unit(col)
        if unit not in units:
            units.append(unit)

    return units


def expand_units_to_features(units, X_columns):
    """
    선택 단위를 실제 모델 입력 변수로 변환.

    Signal4 선택
    → Signal4_Buy, Signal4_Sell 둘 다 포함
    → Signal4_Stay는 기준범주라 제외
    """
    final_features = []

    for unit in units:
        if unit in signal_prefixes:
            for col in [f"{unit}_Buy", f"{unit}_Sell"]:
                if col in X_columns and col not in final_features:
                    final_features.append(col)
        else:
            if unit in X_columns and unit not in final_features:
                final_features.append(unit)

    return final_features


def build_logit_exog(X, features):
    """
    statsmodels 로지스틱 회귀용 design matrix 생성.
    features가 비어 있으면 intercept-only model.
    """
    if len(features) == 0:
        X_model = pd.DataFrame({"const": 1.0}, index=X.index)
    else:
        X_model = X[features].copy()
        X_model = sm.add_constant(X_model, has_constant="add")

    return X_model


def fit_logit_units(X, y, units):
    """
    선택된 unit으로 Logistic Regression 적합.
    추론 기반 선택이므로 statsmodels GLM Binomial 사용.
    """
    features = expand_units_to_features(units, X.columns)
    X_model = build_logit_exog(X, features)

    try:
        model = sm.GLM(y, X_model, family=sm.families.Binomial())
        result = model.fit(maxiter=200, disp=0)

        llf = result.llf
        n = X_model.shape[0]
        k = len(result.params)

        aic = -2 * llf + 2 * k
        bic = -2 * llf + np.log(n) * k

        return {
            "ok": True,
            "units": units.copy(),
            "features": features.copy(),
            "result": result,
            "llf": llf,
            "n_params": k,
            "aic": aic,
            "bic": bic
        }

    except Exception as e:
        return {
            "ok": False,
            "units": units.copy(),
            "features": features.copy(),
            "result": None,
            "llf": -np.inf,
            "n_params": np.nan,
            "aic": np.inf,
            "bic": np.inf,
            "error": str(e)
        }


def lrt_compare(reduced_fit, full_fit):
    """
    Logistic Regression의 reduced model vs full model 비교.
    선형회귀의 partial F-test에 대응되는 로지스틱 회귀의 LRT.
    """
    if (not reduced_fit["ok"]) or (not full_fit["ok"]):
        return {
            "lr_stat": np.nan,
            "df_diff": np.nan,
            "p_value": np.nan
        }

    lr_stat = 2 * (full_fit["llf"] - reduced_fit["llf"])
    lr_stat = max(lr_stat, 0)

    df_diff = full_fit["n_params"] - reduced_fit["n_params"]

    if df_diff <= 0:
        p_value = np.nan
    else:
        p_value = chi2.sf(lr_stat, df_diff)

    return {
        "lr_stat": lr_stat,
        "df_diff": df_diff,
        "p_value": p_value
    }


def forward_selection_logit_lrt(
    X_train,
    y_train,
    entry_alpha=0.05,
    verbose=True
):
    """
    Logistic LRT 기반 Forward Selection.
    예측 성능이 아니라, 변수 추가에 따른 우도 개선 유의성을 기준으로 선택.
    """
    all_units = make_selection_units(X_train.columns)

    selected_units = []
    remaining_units = all_units.copy()

    current_fit = fit_logit_units(X_train, y_train, selected_units)

    history = []
    step = 1

    while len(remaining_units) > 0:
        candidates = []

        for add_unit in remaining_units:
            trial_units = selected_units + [add_unit]
            trial_fit = fit_logit_units(X_train, y_train, trial_units)

            lrt = lrt_compare(current_fit, trial_fit)

            candidates.append({
                "step": step,
                "action": "add_candidate",
                "unit": add_unit,
                "trial_units": trial_units,
                "trial_features": trial_fit["features"],
                "ok": trial_fit["ok"],
                "llf": trial_fit["llf"],
                "aic": trial_fit["aic"],
                "bic": trial_fit["bic"],
                "lr_stat": lrt["lr_stat"],
                "df_diff": lrt["df_diff"],
                "p_value": lrt["p_value"]
            })

        cand_df = pd.DataFrame(candidates)

        valid_cand = cand_df[
            cand_df["ok"] &
            cand_df["p_value"].notna()
        ].copy()

        if len(valid_cand) == 0:
            if verbose:
                print("[Forward] 유효한 후보가 없어 종료")
            break

        valid_cand = valid_cand.sort_values(
            ["p_value", "aic", "bic"],
            ascending=[True, True, True]
        ).reset_index(drop=True)

        best = valid_cand.iloc[0]

        if best["p_value"] < entry_alpha:
            selected_units = best["trial_units"]
            remaining_units.remove(best["unit"])
            current_fit = fit_logit_units(X_train, y_train, selected_units)

            history.append({
                "step": step,
                "action": "add",
                "unit": best["unit"],
                "p_value": best["p_value"],
                "lr_stat": best["lr_stat"],
                "df_diff": best["df_diff"],
                "aic": best["aic"],
                "bic": best["bic"],
                "n_units": len(selected_units),
                "n_features": len(current_fit["features"])
            })

            if verbose:
                print(
                    f"[Forward STEP {step}] add {best['unit']} | "
                    f"p={best['p_value']:.6f} | "
                    f"AIC={best['aic']:.3f} | "
                    f"features={len(current_fit['features'])}"
                )

            step += 1

        else:
            if verbose:
                print(
                    f"[Forward] best p={best['p_value']:.6f} >= entry_alpha={entry_alpha}, 종료"
                )
            break

    final_fit = fit_logit_units(X_train, y_train, selected_units)

    return {
        "method": "Forward_LRT",
        "selected_units": selected_units,
        "selected_features": final_fit["features"],
        "logit_fit": final_fit,
        "history": pd.DataFrame(history)
    }


def backward_selection_logit_lrt(
    X_train,
    y_train,
    remove_alpha=0.10,
    verbose=True
):
    """
    Logistic LRT 기반 Backward Elimination.
    전체 변수에서 시작해서 유의하지 않은 unit을 제거.
    """
    selected_units = make_selection_units(X_train.columns)

    current_fit = fit_logit_units(X_train, y_train, selected_units)

    if not current_fit["ok"]:
        raise RuntimeError(
            "전체 변수 로지스틱 모형 적합 실패. "
            "완전분리 또는 강한 공선성 가능성이 있습니다."
        )

    history = []
    step = 1

    while len(selected_units) > 0:
        candidates = []

        for remove_unit in selected_units:
            reduced_units = [u for u in selected_units if u != remove_unit]

            reduced_fit = fit_logit_units(X_train, y_train, reduced_units)
            lrt = lrt_compare(reduced_fit, current_fit)

            candidates.append({
                "step": step,
                "action": "remove_candidate",
                "unit": remove_unit,
                "reduced_units": reduced_units,
                "reduced_features": reduced_fit["features"],
                "ok": reduced_fit["ok"],
                "reduced_aic": reduced_fit["aic"],
                "reduced_bic": reduced_fit["bic"],
                "current_aic": current_fit["aic"],
                "current_bic": current_fit["bic"],
                "lr_stat": lrt["lr_stat"],
                "df_diff": lrt["df_diff"],
                "p_value": lrt["p_value"]
            })

        cand_df = pd.DataFrame(candidates)

        valid_cand = cand_df[
            cand_df["ok"] &
            cand_df["p_value"].notna()
        ].copy()

        if len(valid_cand) == 0:
            if verbose:
                print("[Backward] 유효한 제거 후보가 없어 종료")
            break

        # p-value가 가장 큰 변수 = 제거해도 손실이 가장 작음
        valid_cand = valid_cand.sort_values(
            ["p_value", "reduced_aic", "reduced_bic"],
            ascending=[False, True, True]
        ).reset_index(drop=True)

        worst = valid_cand.iloc[0]

        if worst["p_value"] > remove_alpha:
            selected_units = worst["reduced_units"]
            current_fit = fit_logit_units(X_train, y_train, selected_units)

            history.append({
                "step": step,
                "action": "remove",
                "unit": worst["unit"],
                "p_value": worst["p_value"],
                "lr_stat": worst["lr_stat"],
                "df_diff": worst["df_diff"],
                "aic": current_fit["aic"],
                "bic": current_fit["bic"],
                "n_units": len(selected_units),
                "n_features": len(current_fit["features"])
            })

            if verbose:
                print(
                    f"[Backward STEP {step}] remove {worst['unit']} | "
                    f"p={worst['p_value']:.6f} | "
                    f"AIC={current_fit['aic']:.3f} | "
                    f"features={len(current_fit['features'])}"
                )

            step += 1

        else:
            if verbose:
                print(
                    f"[Backward] max p={worst['p_value']:.6f} <= remove_alpha={remove_alpha}, 종료"
                )
            break

    final_fit = fit_logit_units(X_train, y_train, selected_units)

    return {
        "method": "Backward_LRT",
        "selected_units": selected_units,
        "selected_features": final_fit["features"],
        "logit_fit": final_fit,
        "history": pd.DataFrame(history)
    }


def stepwise_selection_logit_lrt(
    X_train,
    y_train,
    entry_alpha=0.05,
    remove_alpha=0.10,
    verbose=True
):
    """
    Logistic LRT 기반 Stepwise Selection.
    Forward로 추가하고, 추가 후 Backward로 제거 여부를 점검.
    """
    all_units = make_selection_units(X_train.columns)

    selected_units = []
    remaining_units = all_units.copy()

    current_fit = fit_logit_units(X_train, y_train, selected_units)

    history = []
    step = 1

    while True:
        changed = False

        # =========================
        # Forward step
        # =========================
        candidates = []

        for add_unit in remaining_units:
            trial_units = selected_units + [add_unit]
            trial_fit = fit_logit_units(X_train, y_train, trial_units)
            lrt = lrt_compare(current_fit, trial_fit)

            candidates.append({
                "step": step,
                "phase": "forward",
                "unit": add_unit,
                "trial_units": trial_units,
                "ok": trial_fit["ok"],
                "aic": trial_fit["aic"],
                "bic": trial_fit["bic"],
                "lr_stat": lrt["lr_stat"],
                "df_diff": lrt["df_diff"],
                "p_value": lrt["p_value"]
            })

        cand_df = pd.DataFrame(candidates)

        valid_cand = cand_df[
            cand_df["ok"] &
            cand_df["p_value"].notna()
        ].copy()

        if len(valid_cand) > 0:
            valid_cand = valid_cand.sort_values(
                ["p_value", "aic", "bic"],
                ascending=[True, True, True]
            ).reset_index(drop=True)

            best = valid_cand.iloc[0]

            if best["p_value"] < entry_alpha:
                selected_units = best["trial_units"]
                remaining_units.remove(best["unit"])
                current_fit = fit_logit_units(X_train, y_train, selected_units)

                history.append({
                    "step": step,
                    "action": "add",
                    "unit": best["unit"],
                    "p_value": best["p_value"],
                    "lr_stat": best["lr_stat"],
                    "df_diff": best["df_diff"],
                    "aic": current_fit["aic"],
                    "bic": current_fit["bic"],
                    "n_units": len(selected_units),
                    "n_features": len(current_fit["features"])
                })

                changed = True

                if verbose:
                    print(
                        f"[Stepwise STEP {step}] add {best['unit']} | "
                        f"p={best['p_value']:.6f} | "
                        f"AIC={current_fit['aic']:.3f} | "
                        f"features={len(current_fit['features'])}"
                    )

        # =========================
        # Backward check
        # =========================
        backward_changed = True

        while backward_changed and len(selected_units) > 0:
            backward_changed = False

            current_fit = fit_logit_units(X_train, y_train, selected_units)

            remove_candidates = []

            for remove_unit in selected_units:
                reduced_units = [u for u in selected_units if u != remove_unit]
                reduced_fit = fit_logit_units(X_train, y_train, reduced_units)
                lrt = lrt_compare(reduced_fit, current_fit)

                remove_candidates.append({
                    "step": step,
                    "phase": "backward",
                    "unit": remove_unit,
                    "reduced_units": reduced_units,
                    "ok": reduced_fit["ok"],
                    "reduced_aic": reduced_fit["aic"],
                    "reduced_bic": reduced_fit["bic"],
                    "lr_stat": lrt["lr_stat"],
                    "df_diff": lrt["df_diff"],
                    "p_value": lrt["p_value"]
                })

            remove_df = pd.DataFrame(remove_candidates)

            valid_remove = remove_df[
                remove_df["ok"] &
                remove_df["p_value"].notna()
            ].copy()

            if len(valid_remove) == 0:
                break

            valid_remove = valid_remove.sort_values(
                ["p_value", "reduced_aic", "reduced_bic"],
                ascending=[False, True, True]
            ).reset_index(drop=True)

            worst = valid_remove.iloc[0]

            if worst["p_value"] > remove_alpha:
                selected_units = worst["reduced_units"]
                remaining_units.append(worst["unit"])
                current_fit = fit_logit_units(X_train, y_train, selected_units)

                history.append({
                    "step": step,
                    "action": "remove",
                    "unit": worst["unit"],
                    "p_value": worst["p_value"],
                    "lr_stat": worst["lr_stat"],
                    "df_diff": worst["df_diff"],
                    "aic": current_fit["aic"],
                    "bic": current_fit["bic"],
                    "n_units": len(selected_units),
                    "n_features": len(current_fit["features"])
                })

                changed = True
                backward_changed = True

                if verbose:
                    print(
                        f"[Stepwise STEP {step}] remove {worst['unit']} | "
                        f"p={worst['p_value']:.6f} | "
                        f"AIC={current_fit['aic']:.3f} | "
                        f"features={len(current_fit['features'])}"
                    )

        if not changed:
            if verbose:
                print("[Stepwise] 추가/제거 없음, 종료")
            break

        if len(remaining_units) == 0:
            if verbose:
                print("[Stepwise] 남은 후보 없음, 종료")
            break

        step += 1

    final_fit = fit_logit_units(X_train, y_train, selected_units)

    return {
        "method": "Stepwise_LRT",
        "selected_units": selected_units,
        "selected_features": final_fit["features"],
        "logit_fit": final_fit,
        "history": pd.DataFrame(history)
    }

In [38]:
# =========================
# RF validation evaluator for selected feature sets
# =========================

def rf_valid_metrics_by_threshold(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    gmean = np.sqrt(recall * specificity)

    return {
        "valid_gmean": gmean,
        "valid_accuracy": accuracy_score(y_true, y_pred),
        "valid_f1": f1_score(y_true, y_pred, zero_division=0),
        "valid_precision": precision,
        "valid_recall": recall,
        "valid_specificity": specificity,
        "threshold": threshold,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


def evaluate_feature_set_with_rf(
    X_train,
    y_train,
    X_valid,
    y_valid,
    features,
    thresholds=np.round(np.arange(0.10, 0.91, 0.01), 2),
    random_state=1
):
    """
    이미 선택된 변수셋을 Random Forest에 넣고 valid G-Mean 평가.
    여기서 RF는 변수선택 과정이 아니라, Forward/Backward/Stepwise 결과 비교용 평가기.
    """
    if len(features) == 0:
        return {
            "valid_gmean": np.nan,
            "valid_accuracy": np.nan,
            "valid_f1": np.nan,
            "valid_precision": np.nan,
            "valid_recall": np.nan,
            "valid_specificity": np.nan,
            "threshold": np.nan,
            "tn": np.nan,
            "fp": np.nan,
            "fn": np.nan,
            "tp": np.nan,
            "model": None
        }

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=5,
        min_samples_split=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=random_state,
        n_jobs=-1
    )

    model.fit(X_train[features], y_train)

    y_proba = model.predict_proba(X_valid[features])[:, 1]

    best_metrics = None

    for th in thresholds:
        metrics = rf_valid_metrics_by_threshold(
            y_true=y_valid,
            y_proba=y_proba,
            threshold=th
        )

        if best_metrics is None:
            best_metrics = metrics
        else:
            new_key = (
                metrics["valid_gmean"],
                metrics["valid_recall"],
                metrics["valid_f1"],
                metrics["valid_accuracy"]
            )

            old_key = (
                best_metrics["valid_gmean"],
                best_metrics["valid_recall"],
                best_metrics["valid_f1"],
                best_metrics["valid_accuracy"]
            )

            if new_key > old_key:
                best_metrics = metrics

    best_metrics["model"] = model

    return best_metrics

In [39]:
# =========================
# Run Logistic inference-based selections
# Then compare by RF validation G-Mean
# =========================

entry_alpha = 0.05
remove_alpha = 0.10

print("=" * 80)
print("1. Forward Selection by Logistic LRT")
print("=" * 80)

forward_result = forward_selection_logit_lrt(
    X_train=X_train_scaled,
    y_train=y_train,
    entry_alpha=entry_alpha,
    verbose=True
)

print("\nForward selected units:")
print(forward_result["selected_units"])
print("Forward selected features:")
print(forward_result["selected_features"])


print("\n" + "=" * 80)
print("2. Backward Elimination by Logistic LRT")
print("=" * 80)

backward_result = backward_selection_logit_lrt(
    X_train=X_train_scaled,
    y_train=y_train,
    remove_alpha=remove_alpha,
    verbose=True
)

print("\nBackward selected units:")
print(backward_result["selected_units"])
print("Backward selected features:")
print(backward_result["selected_features"])


print("\n" + "=" * 80)
print("3. Stepwise Selection by Logistic LRT")
print("=" * 80)

stepwise_result = stepwise_selection_logit_lrt(
    X_train=X_train_scaled,
    y_train=y_train,
    entry_alpha=entry_alpha,
    remove_alpha=remove_alpha,
    verbose=True
)

print("\nStepwise selected units:")
print(stepwise_result["selected_units"])
print("Stepwise selected features:")
print(stepwise_result["selected_features"])


# =========================
# RF valid G-Mean으로 세 방법 비교
# =========================

selection_results = [
    forward_result,
    backward_result,
    stepwise_result
]

comparison_rows = []

for res in selection_results:
    method = res["method"]
    units = res["selected_units"]
    features = res["selected_features"]

    rf_metrics = evaluate_feature_set_with_rf(
        X_train=X_train_scaled,
        y_train=y_train,
        X_valid=X_valid_scaled,
        y_valid=y_valid,
        features=features,
        thresholds=np.round(np.arange(0.10, 0.91, 0.01), 2),
        random_state=1
    )

    row = {
        "method": method,
        "selected_units": units,
        "selected_features": features,
        "n_units": len(units),
        "n_features": len(features),
        **{k: v for k, v in rf_metrics.items() if k != "model"}
    }

    comparison_rows.append(row)

method_comparison = pd.DataFrame(comparison_rows)

method_comparison_sorted = method_comparison.sort_values(
    [
        "valid_gmean",
        "valid_recall",
        "valid_f1",
        "valid_accuracy",
        "n_features"
    ],
    ascending=[False, False, False, False, False]
).reset_index(drop=True)

print("\n" + "=" * 80)
print("Forward / Backward / Stepwise RF valid G-Mean 비교")
print("=" * 80)

display(method_comparison_sorted[[
    "method",
    "n_units",
    "n_features",
    "threshold",
    "valid_gmean",
    "valid_recall",
    "valid_specificity",
    "valid_f1",
    "valid_precision",
    "valid_accuracy",
    "tn",
    "fp",
    "fn",
    "tp"
]])

# =========================
# 최종 후보 변수선택 방식 결정
# =========================

best_method_row = method_comparison_sorted.iloc[0]

best_selection_method = best_method_row["method"]
best_selected_units = best_method_row["selected_units"]
best_selected_features = best_method_row["selected_features"]
best_rf_threshold = best_method_row["threshold"]

print("\n" + "=" * 80)
print("최종 선택된 Logistic selection 방식")
print("=" * 80)

print("Best method:", best_selection_method)
print("Best RF threshold:", best_rf_threshold)

print("\n최종 선택 단위:")
print(best_selected_units)

print("\n최종 선택 변수:")
print(best_selected_features)

print("\n최종 선택 변수 개수:", len(best_selected_features))

# 이후 voting에 넣을 변수셋
logistic_selection_features_for_voting = best_selected_features
logistic_selection_units_for_voting = best_selected_units

1. Forward Selection by Logistic LRT
[Forward STEP 1] add NASDAQ_return(%) | p=0.000000 | AIC=1146.753 | features=1
[Forward STEP 2] add VKOSPI_Close | p=0.000001 | AIC=1124.644 | features=2
[Forward STEP 3] add KOSPI 200_DMI14 | p=0.003456 | AIC=1118.094 | features=3
[Forward STEP 4] add VKOSPI_Change(%) | p=0.034232 | AIC=1115.611 | features=4
[Forward STEP 5] add KOSPI 200_OG | p=0.000328 | AIC=1104.707 | features=5
[Forward] best p=0.063825 >= entry_alpha=0.05, 종료

Forward selected units:
['NASDAQ_return(%)', 'VKOSPI_Close', 'KOSPI 200_DMI14', 'VKOSPI_Change(%)', 'KOSPI 200_OG']
Forward selected features:
['NASDAQ_return(%)', 'VKOSPI_Close', 'KOSPI 200_DMI14', 'VKOSPI_Change(%)', 'KOSPI 200_OG']

2. Backward Elimination by Logistic LRT
[Backward STEP 1] remove Signal3 | p=0.777855 | AIC=1120.907 | features=26
[Backward STEP 2] remove Gold Spot_return(%) | p=0.725532 | AIC=1119.031 | features=25
[Backward STEP 3] remove return(%) | p=0.621443 | AIC=1117.275 | features=24
[Backward S

,method,n_units,n_features,threshold,valid_gmean,valid_recall,valid_specificity,valid_f1,valid_precision,valid_accuracy,tn,fp,fn,tp
0,Backward_LRT,8,8,0.23,0.649547,0.629630,0.670093,0.330632,0.224176,0.664773,717,353,60,102
1,Forward_LRT,5,5,0.25,0.639395,0.574074,0.712150,0.330373,0.231920,0.693994,762,308,69,93
2,Stepwise_LRT,5,5,0.25,0.639395,0.574074,0.712150,0.330373,0.231920,0.693994,762,308,69,93



최종 선택된 Logistic selection 방식
Best method: Backward_LRT
Best RF threshold: 0.23

최종 선택 단위:
['KODEX 200_Premium', 'NASDAQ_return(%)', 'KOSPI 200 Volume', 'KOSPI 200 lagged_1_return(%)', 'VKOSPI_Close', 'VKOSPI_Change(%)', 'KOSPI 200_PPO', 'KOSPI 200_OG']

최종 선택 변수:
['KODEX 200_Premium', 'NASDAQ_return(%)', 'KOSPI 200 Volume', 'KOSPI 200 lagged_1_return(%)', 'VKOSPI_Close', 'VKOSPI_Change(%)', 'KOSPI 200_PPO', 'KOSPI 200_OG']

최종 선택 변수 개수: 8


In [40]:
# =========================
# Selection histories
# =========================

print("=" * 80)
print("Forward history")
print("=" * 80)
display(forward_result["history"])

print("\n" + "=" * 80)
print("Backward history")
print("=" * 80)
display(backward_result["history"])

print("\n" + "=" * 80)
print("Stepwise history")
print("=" * 80)
display(stepwise_result["history"])

Forward history


,step,action,unit,p_value,lr_stat,df_diff,aic,bic,n_units,n_features
0,1,add,NASDAQ_return(%),1.065806e-33,146.391849,1,1146.752633,1158.007722,1,1
1,2,add,VKOSPI_Close,9.103319e-07,24.109014,1,1124.643620,1141.526253,2,2
2,3,add,KOSPI 200_DMI14,3.456217e-03,8.549476,1,1118.094144,1140.604321,3,3
3,4,add,VKOSPI_Change(%),3.423234e-02,4.483062,1,1115.611082,1143.748804,4,4
4,5,add,KOSPI 200_OG,3.279296e-04,12.903721,1,1104.707361,1138.472628,5,5



Backward history


,step,action,unit,p_value,lr_stat,df_diff,aic,bic,n_units,n_features
0,1,remove,Signal3,0.777855,0.502431,2,1120.907458,1272.851157,23,26
1,2,remove,Gold Spot_return(%),0.725532,0.123254,1,1119.030713,1265.346867,22,25
2,3,remove,return(%),0.621443,0.243845,1,1117.274558,1257.963167,21,24
3,4,remove,KOSPI 200 lagged_2_return(%),0.565935,0.329532,1,1115.604089,1250.665155,20,23
4,5,remove,KOSPI 200_BB_width,0.557765,0.343589,1,1113.947678,1243.381199,19,22
5,6,remove,USD/KRW_change(%),0.498485,0.458161,1,1112.405839,1236.211815,18,21
6,7,remove,KOSPI 200_PPO_Hist,0.412030,0.672937,1,1111.078775,1229.257208,17,20
7,8,remove,KOSPI 200_RSI14,0.548049,0.360825,1,1109.439600,1221.990488,16,19
8,9,remove,KOSDAQ_return(%),0.354315,0.857943,1,1108.297543,1215.220886,15,18
9,10,remove,KOSPI 200_DMI14,0.334909,0.929824,1,1107.227367,1208.523166,14,17



Stepwise history


,step,action,unit,p_value,lr_stat,df_diff,aic,bic,n_units,n_features
0,1,add,NASDAQ_return(%),1.065806e-33,146.391849,1,1146.752633,1158.007722,1,1
1,2,add,VKOSPI_Close,9.103319e-07,24.109014,1,1124.643620,1141.526253,2,2
2,3,add,KOSPI 200_DMI14,3.456217e-03,8.549476,1,1118.094144,1140.604321,3,3
3,4,add,VKOSPI_Change(%),3.423234e-02,4.483062,1,1115.611082,1143.748804,4,4
4,5,add,KOSPI 200_OG,3.279296e-04,12.903721,1,1104.707361,1138.472628,5,5


In [41]:
# =========================
# 최종 결과 확인 셀
# =========================
# 주의:
# RF valid 성능 비교는 앞 셀에서 이미 수행되어 method_comparison_sorted에 저장되어 있음.
# 여기서는 재계산하지 않고 최종 결과만 확인함.

print("=" * 80)
print("1. Forward / Backward / Stepwise RF valid G-Mean 비교 결과")
print("=" * 80)

display(method_comparison_sorted[[
    "method",
    "n_units",
    "n_features",
    "threshold",
    "valid_gmean",
    "valid_recall",
    "valid_specificity",
    "valid_f1",
    "valid_precision",
    "valid_accuracy",
    "tn",
    "fp",
    "fn",
    "tp"
]])


# =========================
# RF valid G-Mean 기준 최종 채택 방식
# =========================

best_method_row = method_comparison_sorted.iloc[0]

best_selection_method = best_method_row["method"]
best_selected_units = best_method_row["selected_units"]
best_selected_features = best_method_row["selected_features"]
best_rf_threshold = best_method_row["threshold"]

print("\n" + "=" * 80)
print("2. 최종 채택된 Logistic selection 방식")
print("=" * 80)

print("최종 채택 방식:", best_selection_method)
print("RF valid G-Mean:", best_method_row["valid_gmean"])
print("RF best threshold:", best_rf_threshold)

print("\n[최종 선택 단위]")
print(best_selected_units)

print("\n[최종 선택 변수]")
print(best_selected_features)

print("\n최종 선택 변수 개수:", len(best_selected_features))


# =========================
# 방법별 최종 선택 변수 확인
# =========================

print("\n" + "=" * 80)
print("3. 방법별 최종 선택 변수")
print("=" * 80)

print("\n[Forward 선택 변수]")
print(forward_result["selected_features"])
print("변수 개수:", len(forward_result["selected_features"]))

print("\n[Backward 선택 변수]")
print(backward_result["selected_features"])
print("변수 개수:", len(backward_result["selected_features"]))

print("\n[Stepwise 선택 변수]")
print(stepwise_result["selected_features"])
print("변수 개수:", len(stepwise_result["selected_features"]))


# =========================
# Voting에 넣을 변수셋 저장
# =========================

logistic_selection_features_for_voting = best_selected_features
logistic_selection_units_for_voting = best_selected_units
logistic_selection_method_for_voting = best_selection_method

print("\n" + "=" * 80)
print("4. Voting용 변수셋 저장 완료")
print("=" * 80)

print("logistic_selection_method_for_voting:")
print(logistic_selection_method_for_voting)

print("\nlogistic_selection_features_for_voting:")
print(logistic_selection_features_for_voting)


# =========================
# 선택 과정 History 확인
# =========================

print("\n" + "=" * 80)
print("5. Forward history")
print("=" * 80)
display(forward_result["history"])

print("\n" + "=" * 80)
print("6. Backward history")
print("=" * 80)
display(backward_result["history"])

print("\n" + "=" * 80)
print("7. Stepwise history")
print("=" * 80)
display(stepwise_result["history"])

1. Forward / Backward / Stepwise RF valid G-Mean 비교 결과


,method,n_units,n_features,threshold,valid_gmean,valid_recall,valid_specificity,valid_f1,valid_precision,valid_accuracy,tn,fp,fn,tp
0,Backward_LRT,8,8,0.23,0.649547,0.629630,0.670093,0.330632,0.224176,0.664773,717,353,60,102
1,Forward_LRT,5,5,0.25,0.639395,0.574074,0.712150,0.330373,0.231920,0.693994,762,308,69,93
2,Stepwise_LRT,5,5,0.25,0.639395,0.574074,0.712150,0.330373,0.231920,0.693994,762,308,69,93



2. 최종 채택된 Logistic selection 방식
최종 채택 방식: Backward_LRT
RF valid G-Mean: 0.6495465308524644
RF best threshold: 0.23

[최종 선택 단위]
['KODEX 200_Premium', 'NASDAQ_return(%)', 'KOSPI 200 Volume', 'KOSPI 200 lagged_1_return(%)', 'VKOSPI_Close', 'VKOSPI_Change(%)', 'KOSPI 200_PPO', 'KOSPI 200_OG']

[최종 선택 변수]
['KODEX 200_Premium', 'NASDAQ_return(%)', 'KOSPI 200 Volume', 'KOSPI 200 lagged_1_return(%)', 'VKOSPI_Close', 'VKOSPI_Change(%)', 'KOSPI 200_PPO', 'KOSPI 200_OG']

최종 선택 변수 개수: 8

3. 방법별 최종 선택 변수

[Forward 선택 변수]
['NASDAQ_return(%)', 'VKOSPI_Close', 'KOSPI 200_DMI14', 'VKOSPI_Change(%)', 'KOSPI 200_OG']
변수 개수: 5

[Backward 선택 변수]
['KODEX 200_Premium', 'NASDAQ_return(%)', 'KOSPI 200 Volume', 'KOSPI 200 lagged_1_return(%)', 'VKOSPI_Close', 'VKOSPI_Change(%)', 'KOSPI 200_PPO', 'KOSPI 200_OG']
변수 개수: 8

[Stepwise 선택 변수]
['NASDAQ_return(%)', 'VKOSPI_Close', 'KOSPI 200_DMI14', 'VKOSPI_Change(%)', 'KOSPI 200_OG']
변수 개수: 5

4. Voting용 변수셋 저장 완료
logistic_selection_method_for_voting:
Backward_LRT



,step,action,unit,p_value,lr_stat,df_diff,aic,bic,n_units,n_features
0,1,add,NASDAQ_return(%),1.065806e-33,146.391849,1,1146.752633,1158.007722,1,1
1,2,add,VKOSPI_Close,9.103319e-07,24.109014,1,1124.643620,1141.526253,2,2
2,3,add,KOSPI 200_DMI14,3.456217e-03,8.549476,1,1118.094144,1140.604321,3,3
3,4,add,VKOSPI_Change(%),3.423234e-02,4.483062,1,1115.611082,1143.748804,4,4
4,5,add,KOSPI 200_OG,3.279296e-04,12.903721,1,1104.707361,1138.472628,5,5



6. Backward history


,step,action,unit,p_value,lr_stat,df_diff,aic,bic,n_units,n_features
0,1,remove,Signal3,0.777855,0.502431,2,1120.907458,1272.851157,23,26
1,2,remove,Gold Spot_return(%),0.725532,0.123254,1,1119.030713,1265.346867,22,25
2,3,remove,return(%),0.621443,0.243845,1,1117.274558,1257.963167,21,24
3,4,remove,KOSPI 200 lagged_2_return(%),0.565935,0.329532,1,1115.604089,1250.665155,20,23
4,5,remove,KOSPI 200_BB_width,0.557765,0.343589,1,1113.947678,1243.381199,19,22
5,6,remove,USD/KRW_change(%),0.498485,0.458161,1,1112.405839,1236.211815,18,21
6,7,remove,KOSPI 200_PPO_Hist,0.412030,0.672937,1,1111.078775,1229.257208,17,20
7,8,remove,KOSPI 200_RSI14,0.548049,0.360825,1,1109.439600,1221.990488,16,19
8,9,remove,KOSDAQ_return(%),0.354315,0.857943,1,1108.297543,1215.220886,15,18
9,10,remove,KOSPI 200_DMI14,0.334909,0.929824,1,1107.227367,1208.523166,14,17



7. Stepwise history


,step,action,unit,p_value,lr_stat,df_diff,aic,bic,n_units,n_features
0,1,add,NASDAQ_return(%),1.065806e-33,146.391849,1,1146.752633,1158.007722,1,1
1,2,add,VKOSPI_Close,9.103319e-07,24.109014,1,1124.643620,1141.526253,2,2
2,3,add,KOSPI 200_DMI14,3.456217e-03,8.549476,1,1118.094144,1140.604321,3,3
3,4,add,VKOSPI_Change(%),3.423234e-02,4.483062,1,1115.611082,1143.748804,4,4
4,5,add,KOSPI 200_OG,3.279296e-04,12.903721,1,1104.707361,1138.472628,5,5
